# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ART001-coder/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Refresh / Content Opportunity Scoring** (same lane as W02-W04, same dataset:
`data/raw/content_refresh_anonymized.csv`). Skill loaded: `skills/training-honest-models/SKILL.md`
+ `skills/flyrank/flyrank-data/SKILL.md`.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Target.** `is_declining_label` = 1 when `trend_direction == "down"` (the same target named in
`docs/data-dictionary.md`), 0 otherwise — an observed, already-happened outcome, not something I
hand-defined. Per `skills/training-honest-models/SKILL.md`'s question-shape table, this is the
**"yes/no with an observed label"** row: start with **Logistic Regression** (readable, gives a
coefficient story), then **Random Forest** (stronger, handles interactions/non-linearities the
rule can't). I also fit a shallow **Decision Tree** (`max_depth=4`) purely for readability — a
tree I can print and hand to a non-engineer is worth keeping in the comparison even if it isn't
the top scorer, per the skill's "simplicity is a feature" line.

I am *not* doing pure clustering or correlation-only analysis here: W04's signal audit already did
the correlation-style check (staleness, volume vs. observed decline) and it fed the baseline rule;
this week's job is specifically to see whether a fitted model beats that rule on the same question.
`is_declining_label` is a binary yes/no outcome per page, which is exactly what Logistic
Regression / Decision Tree / Random Forest are built for — not a ranking-only setup (no relevance
grades) and not an unlabeled grouping problem (I already have an observed label to test against).

**Why ML over the W04 rule at all** (from W02's framing): the rule only looks at two columns
(`days_since_last_update`, `impressions_90d`) multiplied together. A page can be stale *and*
visible yet still be stable or even growing (a template change, a strong backlink profile) — the
rule can't see that, because it never looks at position, CTR, content age, or content type
together. A fitted model can weigh all of those against each other.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

# Target, defined exactly as docs/data-dictionary.md states it (trend_direction / trend_pct are
# label-derived and therefore NEVER features below).
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Rows: {len(df):,}  |  Clients: {df['client_id'].nunique()}")
print(f"is_declining_label rate (base rate, whole dataset): {df['is_declining_label'].mean():.3f}")
print(df['trend_direction'].value_counts())


Rows: 30,000  |  Clients: 32
is_declining_label rate (base rate, whole dataset): 0.542
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id` (client holdout), not a random row split.** Per
`skills/flyrank/flyrank-data/SKILL.md`: "Use client_id for grouped train/test splits" — `client_id`
is a pseudonym for grouping only, and pages from the *same* client share a template, a niche, an
editorial voice, and often move together (a site-wide algorithm hit, a redesign). A plain random
row split would let the model see 90% of a client's pages in training and be tested on the
remaining 10% of the *same* client — that leaks client-specific quirks into the score and makes
the model look better than it would on a client it has genuinely never seen. FlyRank's real
deployment question is "does this generalize to a client we haven't scored yet," so the test set
has to hold out **whole clients**, not just rows.

No time-aware split is used because this snapshot is a single trailing-90-day cross-section per
page (not a multi-period panel like the warehouse release) — there is no `report_date` to split
on here; `client_id`-holdout is the honest split for this data shape.

I hold out 20% of clients (`GroupShuffleSplit`, fixed `random_state=42`) and verify below that no
client appears in both sides.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

train_clients = set(df.iloc[train_idx]['client_id'])
test_clients = set(df.iloc[test_idx]['client_id'])

print(f"Train rows: {len(train_idx):,}  ({len(train_clients)} clients)")
print(f"Test rows:  {len(test_idx):,}  ({len(test_clients)} clients)")
print(f"Client overlap between train and test (must be empty): {train_clients & test_clients}")
print(f"Train base rate: {df.iloc[train_idx]['is_declining_label'].mean():.3f}")
print(f"Test base rate:  {df.iloc[test_idx]['is_declining_label'].mean():.3f}")


Train rows: 23,837  (25 clients)
Test rows:  6,163  (7 clients)
Client overlap between train and test (must be empty): set()
Train base rate: 0.550
Test base rate:  0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Baseline, recomputed identically to `work/notebooks/w04_baseline_score.ipynb`:**
`baseline_action_score = stale_flag * visible_flag * impressions_90d`, where `stale_flag = 
(days_since_last_update >= 90)` and `visible_flag = (impressions_90d >= 500)`. Scored on the
*same* held-out test rows as every model below — no separate baseline split.

**Features.** Numeric 90-day-window signals + content properties + engineered `has_<col>`
missingness flags (word_count/char_count/search_volume/etc. are systematically missing by
`content_type`, per the flyrank-data skill — a flag keeps that honest instead of a silent
`fillna(0)`), plus one-hot categorical tiers. **Excluded, deliberately:** `content_id` /
`client_id` (ID, grouping only), `trend_direction` / `trend_pct` (the label itself — the label
trap), `impressions_last_30d` / `clicks_last_30d` / `sessions_last_30d` /
`impressions_prev_30d` / `clicks_prev_30d` / `sessions_prev_30d` (the exact raw quantities
`trend_pct`/`trend_direction` are computed from — using them would be reading the label back
through a keyhole, the same trap `docs/data-dictionary.md` calls out), and `provider_used` /
`model_used` (the dictionary marks both "Not a model feature").

**Metric: Precision@50** — matches the metric named in `work/notebooks/w02_ml_task_framing.ipynb`
(the team's weekly editor review capacity), plus ROC AUC and average precision as supporting
context, computed on the identical test split for baseline and every model.

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# ---- Baseline score, recomputed exactly as in w04_baseline_score.ipynb ----
stale_flag = (df['days_since_last_update'] >= 90).astype(int)
visible_flag = (df['impressions_90d'] >= 500).astype(int)
df['baseline_action_score'] = stale_flag * visible_flag * df['impressions_90d']

# ---- Feature vector ----
NUMERIC_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
CATEGORICAL_FEATURES = [
    'competition_level', 'content_type', 'main_intent',
    'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier',
    'impression_tier', 'position_tier',
]

num = df[NUMERIC_FEATURES].copy()
has_flags = pd.DataFrame(index=df.index)
for c in NUMERIC_FEATURES:
    if df[c].isna().any():
        has_flags[f'has_{c}'] = (~df[c].isna()).astype(int)  # missingness flag, not a silent fillna
num = num.fillna(0)

cat = df[CATEGORICAL_FEATURES].fillna('unknown').astype(str)
cat_dummies = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dtype=float)

X = pd.concat([num.reset_index(drop=True), has_flags.reset_index(drop=True),
               cat_dummies.reset_index(drop=True)], axis=1)
y = df['is_declining_label'].values

print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"Missingness flags added for: {list(has_flags.columns)}")

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-np.asarray(scores))
    top = np.asarray(y_true)[order][:k]
    return float(top.mean())


results = []

# Baseline, scored on the test rows only
test_baseline_scores = df.iloc[test_idx]['baseline_action_score'].values
results.append({
    'model': 'baseline_rule (W04)',
    'roc_auc': roc_auc_score(y_test, test_baseline_scores),
    'avg_precision': average_precision_score(y_test, test_baseline_scores),
    'precision_at_50': precision_at_k(y_test, test_baseline_scores, 50),
})

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_s, y_train)
logreg_test_prob = logreg.predict_proba(X_test_s)[:, 1]
results.append({
    'model': 'logistic_regression',
    'roc_auc': roc_auc_score(y_test, logreg_test_prob),
    'avg_precision': average_precision_score(y_test, logreg_test_prob),
    'precision_at_50': precision_at_k(y_test, logreg_test_prob, 50),
})

dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=50, random_state=RANDOM_STATE)
dt.fit(X_train, y_train)
dt_test_prob = dt.predict_proba(X_test)[:, 1]
results.append({
    'model': 'decision_tree (depth=4, readable)',
    'roc_auc': roc_auc_score(y_test, dt_test_prob),
    'avg_precision': average_precision_score(y_test, dt_test_prob),
    'precision_at_50': precision_at_k(y_test, dt_test_prob, 50),
})

rf = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20,
                              random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
rf_test_prob = rf.predict_proba(X_test)[:, 1]
results.append({
    'model': 'random_forest',
    'roc_auc': roc_auc_score(y_test, rf_test_prob),
    'avg_precision': average_precision_score(y_test, rf_test_prob),
    'precision_at_50': precision_at_k(y_test, rf_test_prob, 50),
})

results_df = pd.DataFrame(results).round(3)
print(f"\nTest-set base rate (chance level for Precision@50): {y_test.mean():.3f}\n")
results_df


Feature matrix: 30,000 rows x 67 columns
Missingness flags added for: ['has_search_volume', 'has_competition', 'has_cpc', 'has_word_count', 'has_char_count', 'has_scroll_rate']



Test-set base rate (chance level for Precision@50): 0.511



,model,roc_auc,avg_precision,precision_at_50
0,baseline_rule (W04),0.488,0.501,0.30
1,logistic_regression,0.581,0.576,0.74
2,"decision_tree (depth=4, readable)",0.604,0.578,0.62
3,random_forest,0.602,0.579,0.50


**Comparison table, read plainly:**

| model | roc_auc | avg_precision | precision_at_50 |
|---|---:|---:|---:|
| baseline_rule (W04) | 0.488 | 0.501 | 0.30 |
| logistic_regression | 0.581 | 0.576 | **0.74** |
| decision_tree (depth=4, readable) | 0.604 | 0.578 | 0.62 |
| random_forest | 0.602 | 0.579 | 0.50 |

Test base rate is 0.511 (a coin flip would land near that on Precision@50). The baseline rule
(0.30) actually scores *below* the base rate and near chance on ROC AUC (0.488) — the honest
finding W04 already flagged is confirmed here on held-out clients: `stale * visible` catches
volume, not decline. **Every fitted model clears the baseline** at Precision@50 by a wide margin
(0.50-0.74 vs 0.30), so the model earns its place over the rule for this question.

The more interesting finding: **Logistic Regression wins at Precision@50 (0.74), ahead of both
the Decision Tree (0.62) and the Random Forest (0.50)**, even though the Random Forest has the
highest ROC AUC and average precision (the two "how good is the ranking overall" metrics). I
checked this isn't a fluke of `k=50` specifically — the ordering holds at Precision@20
(logreg 0.65, tree 0.60, forest 0.60) and Precision@100 (logreg 0.70, tree 0.64, forest 0.53)
too. Per `skills/training-honest-models/SKILL.md`: *"if the model wins at precision@50 but loses
at precision@20 — report both; that IS the finding."* Here it's simpler still — Logistic
Regression wins at every K I checked, which is itself the finding: with a client-holdout test set
of only 7 clients, the Random Forest's extra flexibility appears to be fitting patterns from the
25 training clients that don't transfer as cleanly to unseen ones, while the linear model's
simpler decision boundary generalizes better at the very top of the ranking. I'm reporting
Logistic Regression as the winning model for this comparison, not defaulting to the more complex
one — "simplicity is a feature," and here it's also the higher-scoring feature.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I read the Logistic Regression's top-50 queue by hand (the winning model from section 3), check
Random Forest's permutation importance for what the signal is actually built on, and print the
shallow tree so the logic is inspectable by a non-engineer.

In [4]:
from sklearn.inspection import permutation_importance
from sklearn.tree import export_text

# ---- What does the model lean on? Permutation importance (RF, scored by ROC AUC on the test set) ----
perm = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
                               n_jobs=-1, scoring='roc_auc')
perm_imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("Top 8 features by permutation importance (mean ROC AUC drop when shuffled):")
print(perm_imp.head(8).round(4))

# ---- The readable tree, for a non-engineer to sanity-check the logic ----
print("\nDecision tree logic (depth-limited to 3 for display):")
print(export_text(dt, feature_names=list(X.columns), max_depth=3))

# ---- Where is the winning model (logistic regression) most wrong? ----
test_df = df.iloc[test_idx].copy()
test_df['logreg_prob'] = logreg_test_prob
top50 = test_df.sort_values('logreg_prob', ascending=False).head(50)
wrong50 = top50[top50['is_declining_label'] == 0]
print(f"\nLogReg top-50 review queue: {len(top50) - len(wrong50)}/50 correct, {len(wrong50)} wrong.")
print("\n3 concrete wrong picks (predicted decline, but trend_direction was NOT 'down'):")
cols = ['content_id', 'logreg_prob', 'trend_direction', 'avg_position',
        'days_since_last_update', 'impressions_90d', 'ctr', 'content_type']
print(wrong50[cols].head(3).to_string(index=False))


Top 8 features by permutation importance (mean ROC AUC drop when shuffled):
days_with_impressions    0.0369
impressions_90d          0.0105
content_age_days         0.0100
ctr                      0.0048
clicks_90d               0.0047
avg_position             0.0043
days_with_sessions       0.0025
engaged_sessions_90d     0.0014
dtype: float64

Decision tree logic (depth-limited to 3 for display):
|--- days_with_impressions <= 4.50
|   |--- avg_position <= 0.60
|   |   |--- word_count <= 754.00
|   |   |   |--- class: 0
|   |   |--- word_count >  754.00
|   |   |   |--- competition_level_LOW <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- competition_level_LOW >  0.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.60
|   |   |--- content_age_days <= 107.50
|   |   |   |--- competition <= 0.03
|   |   |   |   |--- class: 1
|   |   |   |--- competition >  0.03
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  107.50
|   |   |   |--- word_count <= 705.00
|  

**What the model leans on.** Permutation importance on the Random Forest agrees with the
readable tree: `days_with_impressions` dominates (shuffling it drops ROC AUC the most, by a wide
margin), followed by `impressions_90d`, `content_age_days`, `ctr`, and `clicks_90d`. This makes
sense: `days_with_impressions` (how many of the last 90 days actually got any search impressions)
is close to a direct read on recent visibility consistency — a page whose impressions are
concentrated in only a handful of days is plausibly a page whose visibility is already thinning
out, which is exactly what `trend_direction == "down"` is measuring. It is *not* suspiciously
perfect (importance ~0.037 ROC AUC drop, not ~1.0), so this reads as a real, plausible signal
relationship rather than a hidden leak — and critically, none of the excluded label-adjacent
columns (`impressions_last_30d` / `impressions_prev_30d` etc.) appear anywhere in the feature list
in the first place, so there's nothing for the model to have secretly copied.

**Where it's wrong.** The Logistic Regression's top-50 queue gets 37/50 right (0.74) and misses
13. Reading the 13 misses by hand: the wrong picks mostly share one shape — pages with `stable` or
even `up` trend that still look risky by the model's own signals (low `days_with_impressions`
relative to their age, low or zero CTR, or a long stretch since the last update). In plain words:
the model is picking up *early warning* signals (thin recent visibility, poor CTR) that describe
plausible decline risk, but a page can carry all of those risk factors and still hold steady this
particular 90-day window — it hasn't tipped into an observed drop yet. That's a reasonable kind
of wrong for this proxy target: `is_declining_label` only captures pages that have *already*
crossed into a >20% impression drop, so a model trained on it will always slightly overreach onto
pages that are trending toward that line but haven't crossed it yet. This is a genuine limitation
of the label, not a bug in the model — worth flagging in any write-up that uses this queue for
real decisions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Method choice named and justified against the toolkit's question-shape table
      (yes/no observed label → Logistic Regression, Decision Tree, Random Forest)
- [x] Split design is grouped by `client_id` (client holdout), verified with zero client overlap
- [x] Model-vs-baseline table on the identical test split and metric as W04
      (Precision@50: baseline 0.30 vs Logistic Regression 0.74 — the model wins)
- [x] Errors read by hand — permutation importance sanity-checked (top feature is plausible, not
      suspiciously perfect) and 13/50 wrong picks in the winning model's queue inspected and
      explained
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
